# Build Your MCP Server Using Bedrock AgentCore Gateway
## MCPify your AWS Lambda functions as Gateway targets

### The Scenario

You already have **APIs running as AWS Lambda functions**. You want Amazon Quick agents to invoke them, but you don't want to build and host an MCP server.

**AgentCore Gateway** solves this. It wraps your existing Lambda functions and exposes them as MCP tools automatically. You just define tool schemas (name, description, input parameters) and Gateway handles the MCP protocol, authentication, and routing.

### This Pattern Works for Any Lambda

While this workshop uses **HR APIs as an example**, the pattern is identical for any Lambda function:
- Order management, CRM, inventory, billing, ticketing
- ISV product APIs, enterprise integrations, custom services
- Any business logic you already run on Lambda

### What You'll Deploy (Cell by Cell)

| Step | What You Create | Why |
|------|----------------|-----|
| **1** | Install dependencies | SDK and libraries |
| **2** | 4 AWS Lambda functions | Your existing APIs (one per HR operation) |
| **3** | IAM role for Gateway | Outbound auth — Gateway assumes this role to invoke your Lambdas |
| **4** | Cognito user pool, domain, scopes, client | Inbound auth — validates OAuth tokens from Amazon Quick |
| **5** | AgentCore Gateway with Cognito authorizer | The MCP endpoint that Quick connects to |
| **6** | 4 Lambda targets with tool schemas | Maps each Lambda to an MCP tool definition |
| **7** | Test tools through Gateway | Verify everything works end-to-end |
| **8** | Connection details for Quick | Gateway URL, Client ID, Client Secret, Token URL |

### Architecture
```
                                                    ┌──▶ hr-gw-get-payroll
Amazon Quick  ──OAuth──▶  AgentCore Gateway  ──IAM──┼──▶ hr-gw-get-org-chart
  (MCP client)              (MCP server)            ├──▶ hr-gw-submit-timesheet
                                                    └──▶ hr-gw-get-benefits
```

**⚠️ Run cells one at a time with Shift+Enter.**

### Prerequisites
- SageMaker JupyterLab with an execution role that has `AdministratorAccess`

---
## Step 1: Install Dependencies
⏱️ ~1 minute

- `boto3` — AWS SDK for creating Lambda, IAM, Cognito, and Gateway resources
- `requests` — HTTP client for OAuth token requests

In [ ]:
!pip install -q boto3 requests
print('\n✅ Dependencies installed')

---
## Step 2: Create the HR Lambda Functions
⏱️ ~30 seconds (includes IAM role propagation)

This creates **4 separate Lambda functions**, one per HR operation:

| Lambda Function | Operation | Description |
|----------------|-----------|-------------|
| `hr-gw-get-payroll` | `get_payroll_info` | Retrieve employee salary and compensation |
| `hr-gw-get-org-chart` | `get_org_chart` | Query reporting structure and direct reports |
| `hr-gw-submit-timesheet` | `submit_timesheet` | Submit weekly timesheet hours |
| `hr-gw-get-benefits` | `get_benefits_summary` | View employee benefits enrollment |

Each Lambda is a simple, focused function that handles one operation. Gateway routes tool invocations to the correct Lambda automatically.

**In your real scenario**, these Lambdas already exist — they're your existing APIs. You'd skip this step and just point Gateway at your Lambda ARNs.

In [ ]:
import os
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'us-east-1')

from utils import create_all_hr_lambdas

lambda_arns = create_all_hr_lambdas()

print(f'\n✅ All 4 Lambda functions deployed:')
for name, arn in lambda_arns.items():
    print(f'  {name}: {arn}')

---
## Step 3: Create IAM Role for Gateway (Outbound Auth)
⏱️ ~10 seconds

AgentCore Gateway needs permission to **invoke your Lambda functions** on behalf of authenticated users. This is the **outbound authorization** — Gateway assumes this IAM role when calling your Lambdas.

The trust policy allows `bedrock-agentcore.amazonaws.com` to assume this role, and the permissions policy grants `lambda:InvokeFunction` on all `hr-gw-*` functions.

In [ ]:
from utils import create_agentcore_gateway_role

gateway_role_arn = create_agentcore_gateway_role()
print(f'\n✅ Gateway Role ARN: {gateway_role_arn}')

---
## Step 4: Set Up Cognito Authentication (Inbound Auth)
⏱️ ~10 seconds

This is the **inbound authentication** — when Amazon Quick calls your Gateway, it sends an OAuth access token. Gateway validates this token against your Cognito user pool before allowing access.

Creates:

| Resource | Why |
|----------|-----|
| **User Pool** | Container for OAuth configuration |
| **Cognito Domain** | Creates the `/oauth2/token` endpoint that Quick calls to get tokens |
| **Resource Server + Scopes** | Defines `hr-gateway/read` and `hr-gateway/write` scopes |
| **App Client with Secret** | Quick sends `client_id` + `client_secret` to get a JWT token |

In [ ]:
from utils import setup_cognito_for_gateway

cognito_config = setup_cognito_for_gateway()

print(f'\n✓ Pool ID:        {cognito_config["pool_id"]}')
print(f'✓ Client ID:      {cognito_config["client_id"]}')
print(f'✓ Client Secret:  {cognito_config["client_secret"][:10]}...')
print(f'✓ Token URL:      {cognito_config["token_url"]}')
print(f'✓ Discovery URL:  {cognito_config["discovery_url"]}')
print(f'\n✅ Cognito setup complete')

---
## Step 5: Create AgentCore Gateway
⏱️ ~30 seconds

Now we create the **AgentCore Gateway** itself. This is the MCP endpoint that Amazon Quick will connect to.

The Gateway is configured with:
- **Protocol**: MCP — Gateway automatically exposes `ListTools` and `InvokeTools` APIs
- **Authorizer**: Custom JWT — validates OAuth tokens from Cognito
- **Role**: The IAM role from Step 3 — used for outbound Lambda invocation

After this cell, you'll have a Gateway URL — this is your MCP server endpoint, created without writing any MCP server code.

In [ ]:
import boto3
from botocore.exceptions import ClientError

gateway_client = boto3.client('bedrock-agentcore-control', region_name=os.environ['AWS_DEFAULT_REGION'])

auth_config = {
    'customJWTAuthorizer': {
        'allowedClients': [cognito_config['client_id']],
        'discoveryUrl': cognito_config['discovery_url']
    }
}

try:
    create_response = gateway_client.create_gateway(
        name='HRGatewayMCPv2',
        roleArn=gateway_role_arn,
        protocolType='MCP',
        authorizerType='CUSTOM_JWT',
        authorizerConfiguration=auth_config,
        description='HR payroll and org tools via AgentCore Gateway'
    )
    gateway_id = create_response['gatewayId']
    gateway_url = create_response['gatewayUrl']
    print(f'✓ Gateway created')
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        gateways = gateway_client.list_gateways()
        for gw in gateways['items']:
            if gw['name'] == 'HRGatewayMCPv2':
                gateway_id = gw['gatewayId']
                gateway_url = gw['gatewayUrl']
                break
        print(f'ℹ️  Gateway already exists, reusing')
    else:
        raise

print(f'✓ Gateway ID:  {gateway_id}')
print(f'✓ Gateway URL: {gateway_url}')
print(f'\n✅ AgentCore Gateway ready — this is your MCP endpoint')

---
## Step 6: Add Lambda Functions as Gateway Targets with MCP Tool Schemas
⏱️ ~10 seconds

This is the **key step** — you register each Lambda function as a Gateway target and define its **MCP tool schema**. Each target maps one Lambda to one MCP tool.

The schemas tell Amazon Quick:
- What tools are available (name + description)
- What input parameters each tool accepts (JSON Schema)

Gateway uses these schemas to:
1. Respond to `ListTools` — Quick discovers your tools
2. Route `InvokeTools` calls — Gateway invokes the correct Lambda with the arguments

**To adapt for your own Lambdas**: Replace the Lambda ARNs, tool names, descriptions, and input schemas below.

In [ ]:
import time
from botocore.exceptions import ClientError
time.sleep(5)  # Wait for Gateway to be fully ready

tool_targets = [
    {
        'target_name': 'GetPayroll',
        'lambda_key': 'hr-gw-get-payroll',
        'tool_name': 'get_payroll_info',
        'description': 'Retrieve employee payroll and compensation details including salary, pay frequency, and annual compensation',
        'schema': {
            'type': 'object',
            'properties': {'employee_id': {'type': 'string', 'description': 'Employee ID (e.g., EMP001)'}},
            'required': ['employee_id']
        }
    },
    {
        'target_name': 'GetOrgChart',
        'lambda_key': 'hr-gw-get-org-chart',
        'tool_name': 'get_org_chart',
        'description': 'Query the organizational reporting structure including manager and direct reports for an employee',
        'schema': {
            'type': 'object',
            'properties': {'employee_id': {'type': 'string', 'description': 'Employee ID (e.g., EMP001)'}},
            'required': ['employee_id']
        }
    },
    {
        'target_name': 'SubmitTimesheet',
        'lambda_key': 'hr-gw-submit-timesheet',
        'tool_name': 'submit_timesheet',
        'description': 'Submit weekly timesheet hours for an employee',
        'schema': {
            'type': 'object',
            'properties': {
                'employee_id': {'type': 'string', 'description': 'Employee ID (e.g., EMP001)'},
                'week_ending': {'type': 'string', 'description': 'Week ending date in YYYY-MM-DD format'},
                'hours_worked': {'type': 'number', 'description': 'Total hours worked for the week'}
            },
            'required': ['employee_id', 'week_ending', 'hours_worked']
        }
    },
    {
        'target_name': 'GetBenefits',
        'lambda_key': 'hr-gw-get-benefits',
        'tool_name': 'get_benefits_summary',
        'description': 'View employee benefits enrollment including health, dental, vision, retirement, and PTO policy',
        'schema': {
            'type': 'object',
            'properties': {'employee_id': {'type': 'string', 'description': 'Employee ID (e.g., EMP001)'}},
            'required': ['employee_id']
        }
    }
]

credential_config = [{'credentialProviderType': 'GATEWAY_IAM_ROLE'}]
target_ids = {}

for tool in tool_targets:
    target_config = {
        'mcp': {
            'lambda': {
                'lambdaArn': lambda_arns[tool['lambda_key']],
                'toolSchema': {
                    'inlinePayload': [{
                        'name': tool['tool_name'],
                        'description': tool['description'],
                        'inputSchema': tool['schema']
                    }]
                }
            }
        }
    }

    try:
        resp = gateway_client.create_gateway_target(
            gatewayIdentifier=gateway_id,
            name=tool['target_name'],
            description=tool['description'],
            targetConfiguration=target_config,
            credentialProviderConfigurations=credential_config
        )
        target_ids[tool['target_name']] = resp['targetId']
        print(f"✓ {tool['target_name']} → {tool['tool_name']} ({tool['lambda_key']})")
    except ClientError as e:
        if e.response['Error']['Code'] == 'ConflictException':
            print(f"ℹ️  {tool['target_name']} already exists, skipping")
        else:
            raise

print(f'\n✅ 4 Lambda targets registered with MCP tool schemas')

---
## Step 7: Test — Invoke Tools Through the Gateway
⏱️ ~30 seconds

Let's verify everything works end-to-end:
1. Get an OAuth token from Cognito (simulating what Amazon Quick does)
2. Call the Gateway MCP endpoint to invoke each tool

This confirms the full auth chain: OAuth token → Gateway validates → IAM role → correct Lambda invoked.

In [ ]:
import time
import json
from utils import get_oauth_token

print('Waiting for Cognito domain propagation...')
time.sleep(15)

token = get_oauth_token(cognito_config)
print(f'✓ OAuth token obtained: {token[:20]}...')

test_cases = [
    ('GetPayroll', 'get_payroll_info', {'employee_id': 'EMP001'}),
    ('GetOrgChart', 'get_org_chart', {'employee_id': 'EMP001'}),
    ('SubmitTimesheet', 'submit_timesheet', {'employee_id': 'EMP001', 'week_ending': '2026-02-13', 'hours_worked': 40}),
    ('GetBenefits', 'get_benefits_summary', {'employee_id': 'EMP002'}),
]

print(f'\nTesting 4 tools via Gateway:\n')

data_client = boto3.client('bedrock-agentcore', region_name=os.environ['AWS_DEFAULT_REGION'])

for target_name, tool_name, args in test_cases:
    full_tool_name = f'{target_name}___{tool_name}'
    print(f'── {tool_name} ──')
    try:
        result = data_client.invoke_tool(
            gatewayIdentifier=gateway_id,
            toolName=full_tool_name,
            toolInputs=args,
            authorizationToken=f'Bearer {token}'
        )
        body = result.get('body', result)
        if isinstance(body, str):
            body = json.loads(body)
        print(f'  ✓ {json.dumps(body, indent=4)}')
    except Exception as e:
        print(f'  ✗ Error: {e}')
    print()

print('✅ All tools tested successfully')

---
## Step 8: 📋 Connection Details for Amazon Quick

Copy these 4 values into the Amazon Quick MCP integration setup.

In Quick: **Integrations → Actions → Model Context Protocol → +**

In [ ]:
print('=' * 60)
print('📋 GATEWAY MCP CONNECTION DETAILS')
print('   Copy these into the Amazon Quick MCP Client')
print('=' * 60)
print()
print(f'MCP Server URL:   {gateway_url}')
print(f'Client ID:        {cognito_config["client_id"]}')
print(f'Client Secret:    {cognito_config["client_secret"]}')
print(f'Token URL:        {cognito_config["token_url"]}')
print()
print('=' * 60)
print('Gateway ID (for cleanup): ', gateway_id)
print('=' * 60)

---
## ✅ Workshop Complete!

### What you deployed
- **4 Lambda functions** — one per HR operation (payroll, org chart, timesheets, benefits)
- **IAM role** — Outbound auth for Gateway to invoke Lambdas
- **Cognito** — User Pool + domain + scopes + app client for inbound OAuth
- **AgentCore Gateway** — MCP endpoint with Cognito authorizer
- **4 Gateway targets** — each Lambda registered with its MCP tool schema

### Key takeaway
You turned existing Lambda functions into MCP tools **without writing any MCP server code**. Gateway handled the MCP protocol, authentication, and routing to the correct Lambda.

### Adapt for your own APIs
To expose your own Lambda functions:
1. Replace the Lambda ARNs with your own
2. Update the tool schemas in Step 6 with your tool names, descriptions, and input schemas
3. That's it — Gateway does the rest

### Next step
→ Connect to Amazon Quick using the credentials from Step 8

In [ ]:
# OPTIONAL: Cleanup all resources
# Uncomment and run when you're done with the workshop

# from utils import delete_gateway
# delete_gateway(gateway_client, gateway_id)

# lambda_client = boto3.client('lambda', region_name=os.environ['AWS_DEFAULT_REGION'])
# for name in lambda_arns:
#     lambda_client.delete_function(FunctionName=name)
#     print(f'✓ Deleted Lambda: {name}')

# iam = boto3.client('iam')
# iam.delete_role_policy(RoleName='hr-agentcore-gateway-role', PolicyName='gateway-lambda-invoke')
# iam.delete_role(RoleName='hr-agentcore-gateway-role')
# iam.detach_role_policy(RoleName='hr-gateway-lambda-role', PolicyArn='arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole')
# iam.delete_role(RoleName='hr-gateway-lambda-role')
# print('✓ IAM roles deleted')

# cognito = boto3.client('cognito-idp')
# cognito.delete_user_pool_domain(UserPoolId=cognito_config['pool_id'], Domain=cognito_config['token_url'].split('//')[1].split('.')[0])
# cognito.delete_user_pool(UserPoolId=cognito_config['pool_id'])
# print('✓ Cognito resources deleted')